# 3 · Post-BERTopic processing

Joins the BERTopic output (`topics_documents_0218.pkl`) with the bibliographic export (`df.xlsx`, row order preserved) and builds the author–year reference keys used to trace citations in the synthesis.

* **Output:** `df_merged_postbertopic.xlsx` / `.pkl`.
* **Environment:** Google Colab; paths point to the authors' Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_excel('/content/drive/MyDrive/Phd(c)/REVIEW_SEGURIDAD/CRUCE BERTOPIC/df.xlsx')

In [ ]:
df_bertopic = pd.read_pickle('/content/drive/MyDrive/Phd(c)/REVIEW_SEGURIDAD/CRUCE BERTOPIC/topics_documents_0218.pkl')

In [ ]:
df.columns

In [ ]:
df_bertopic.columns

In [ ]:
# Unir los DataFrames por índice
df_merged = pd.concat([df, df_bertopic], axis=1)


In [ ]:
df_merged.columns

In [ ]:
df_merged["Authors"]

## Author–year reference keys

In [ ]:
import ast

# Función para formatear la referencia
def format_reference(authors, year):
    if isinstance(authors, str):
        if authors.startswith("["):
            # Convertir la cadena en una lista real
            try:
                authors_list = ast.literal_eval(authors)
                first_author = authors_list[0] if authors_list else "Desconocido"
            except (SyntaxError, ValueError):
                return "[Desconocido et al., {}]".format(year) if pd.notna(year) else "[Desconocido et al.]"
        else:
            first_author = authors.split(';')[0].strip()  # Tomar el primer autor en formato "Apellido, Nombre"
    elif isinstance(authors, list):  # Si ya es lista
        first_author = authors[0] if authors else "Desconocido"
    else:
        return "[Desconocido et al., {}]".format(year) if pd.notna(year) else "[Desconocido et al.]"

    # Extraer solo el apellido
    last_name = first_author.split(',')[0] if ',' in first_author else first_author.split()[-1]

    return f"[{last_name} et al., {year}]" if pd.notna(year) else f"[{last_name} et al.]"

# Aplicar la función a cada fila
df_merged['Reference'] = df_merged.apply(lambda row: format_reference(row['Authors'], row['Publication Year']), axis=1)


In [ ]:
df_merged['Reference']

In [ ]:
df_merged

In [ ]:
df_merged.to_excel('/content/drive/MyDrive/Phd(c)/REVIEW_SEGURIDAD/CRUCE BERTOPIC/df_merged_postbertopic.xlsx', index=False)

In [ ]:
df_merged.to_pickle('/content/drive/MyDrive/Phd(c)/REVIEW_SEGURIDAD/CRUCE BERTOPIC/df_merged_postbertopic.pkl')